# Dataset Budget

## Import Semua Packages/Library yang Digunakan

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import re

## Data Wrangling

### Gathering Data

In [2]:
# load dataset budget
file_id = "11xx2Wvlx0KUbBi2bBqVo6gYZiaLd7f6g"
url = f"https://drive.google.com/uc?export=download&id={file_id}"

df = pd.read_csv(url)

# melihat data awal
df.head()

,budget_id,user_id,income_id,needs_amount,wants_amount,investment_amount,income_amount,budget_limit,source,income_date
0,BGT01352,USR053,INC06728,3400000,2250000,1600000,7250000,5650000,salary,2023-01-10 00:38
1,BGT06784,USR016,INC00797,650000,400000,300000,1400000,1050000,scholarship,2024-05-25 12:49
2,BGT05527,USR050,INC01634,2150000,1150000,550000,3800000,3300000,freelance,2023-04-25 15:24
3,BGT02948,USR013,INC08009,600000,350000,250000,1200000,950000,scholarship,2023-07-21 06:16
4,BGT03140,USR038,INC06623,2550000,1450000,950000,4950000,4000000,freelance,2024-04-07 04:12


### Asessing Data

In [3]:
# cek dimensi dataset
df.shape

(10040, 10)

In [4]:
# cek tipe data dan jumlah non-null tiap kolom
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10040 entries, 0 to 10039
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   budget_id          10040 non-null  object
 1   user_id            10040 non-null  object
 2   income_id          10040 non-null  object
 3   needs_amount       9830 non-null   object
 4   wants_amount       9826 non-null   object
 5   investment_amount  9706 non-null   object
 6   income_amount      9778 non-null   object
 7   budget_limit       10040 non-null  int64 
 8   source             9934 non-null   object
 9   income_date        9759 non-null   object
dtypes: int64(1), object(9)
memory usage: 784.5+ KB


In [5]:
# cek missing values (NaN) per kolom
df.isnull().sum()

,0
budget_id,0
user_id,0
income_id,0
needs_amount,210
wants_amount,214
investment_amount,334
income_amount,262
budget_limit,0
source,106
income_date,281


In [6]:
# cek duplikasi baris penuh
print("Jumlah baris duplikat:", df.duplicated().sum())

Jumlah baris duplikat: 40


In [7]:
# cek inkonsistensi nilai kategorikal
for col in ['source']:
    print(f"\n{col}:", df[col].unique())


source: ['salary' 'scholarship' 'freelance' 'FREELANCE' 'business' 'bonus'
 'allowance' 'part_time' 'investment_return' 'internship' nan 'PART_TIME'
 'SALARY' 'ALLOWANCE' 'INVESTMENT_RETURN' 'BONUS' 'SCHOLARSHIP' 'BUSINESS'
 'INTERNSHIP']


In [8]:
# frekuensi source
print(df['source'].value_counts())

source
freelance            1698
allowance            1635
part_time            1462
business             1036
salary               1029
bonus                 987
scholarship           828
internship            648
investment_return     412
ALLOWANCE              47
PART_TIME              37
FREELANCE              31
SALARY                 17
BONUS                  17
SCHOLARSHIP            17
BUSINESS               13
INVESTMENT_RETURN      12
INTERNSHIP              8
Name: count, dtype: int64


In [9]:
# cek statistik deskriptif data numerik
df.describe()

,budget_limit
count,1.004000e+04
mean,3.419895e+06
std,1.826738e+06
min,7.000000e+05
25%,1.900000e+06
50%,3.100000e+06
75%,4.750000e+06
max,9.850000e+06


In [10]:
# Deteksi inkonsistensi format tanggal
date_sample = df['income_date'].dropna().astype(str)

fmt_ddmmyyyy   = date_sample.str.match(r'^\d{2}/\d{2}/\d{4}').sum()
fmt_yyyymmdd_s = date_sample.str.match(r'^\d{4}/\d{2}/\d{2}').sum()
fmt_yyyymmdd_d = date_sample.str.match(r'^\d{4}-\d{2}-\d{2}').sum()

print(f'DD/MM/YYYY HH:MM : {fmt_ddmmyyyy} baris')
print(f'YYYY/MM/DD HH:MM:SS: {fmt_yyyymmdd_s} baris')
print(f'YYYY-MM-DD HH:MM:SS (standar) : {fmt_yyyymmdd_d} baris')

DD/MM/YYYY HH:MM : 268 baris
YYYY/MM/DD HH:MM:SS: 158 baris
YYYY-MM-DD HH:MM:SS (standar) : 9333 baris


In [11]:
# Deteksi data numerik tidak valid
numeric_cols = [
    'needs_amount', 'wants_amount', 'investment_amount',
    'income_amount', 'budget_limit'
]

for col in numeric_cols:
    non_numeric = df[pd.to_numeric(df[col], errors='coerce').isna() & df[col].notna()]
    print(f'Kolom {col} - data non numerik : {len(non_numeric)}')

Kolom needs_amount - data non numerik : 136
Kolom wants_amount - data non numerik : 142
Kolom investment_amount - data non numerik : 161
Kolom income_amount - data non numerik : 161
Kolom budget_limit - data non numerik : 0


In [12]:
# cek apakah total budget melebihi income
invalid_budget = df[
    (
        pd.to_numeric(df['needs_amount'], errors='coerce') +
        pd.to_numeric(df['wants_amount'], errors='coerce') +
        pd.to_numeric(df['investment_amount'], errors='coerce')
    ) > pd.to_numeric(df['income_amount'], errors='coerce')
]

print("Jumlah budget melebihi income:", len(invalid_budget))
invalid_budget.head()


Jumlah budget melebihi income: 1107


,budget_id,user_id,income_id,needs_amount,wants_amount,investment_amount,income_amount,budget_limit,source,income_date
2,BGT05527,USR050,INC01634,2150000,1150000,550000,3800000,3300000,freelance,2023-04-25 15:24
63,BGT05707,USR049,INC03730,3900000,2250000,1650000,7750000,6150000,salary,2024-08-31 03:02
65,BGT08101,USR022,INC05222,1700000,1100000,900000,3650000,2800000,NaN,2024-09-06 15:36
67,BGT03018,USR029,INC02343,1900000,1300000,850000,4000000,3200000,part_time,2023-05-26 06:53
79,BGT08435,USR015,INC00156,1400000,950000,700000,3000000,2350000,allowance,2024-02-15 01:37


**Insight:**
- Terdapat missing values pada beberapa kolom numerik dan kategorikal
- Terdapat beberapa baris duplikat pada dataset
- Kolom source memiliki typo dan inkonsistensi penulisan
- Kolom income_date menggunakan format datetime yang tidak konsisten
- Beberapa kolom numerik masih berbentuk string dan perlu dibersihkan
- Ditemukan nilai negatif dan nol pada beberapa nominal budget
- Terdapat inkonsistensi antara total budget dan income_amount
- Ditemukan outlier pada income_amount

### Cleaning Data


In [13]:
# Membuat salinan agar data asli tetap terjaga
df_clean = df.copy()

In [14]:
# Bersihkan & konversi kolom numerik
numeric_cols = [
    'needs_amount', 'wants_amount', 'investment_amount',
    'income_amount', 'budget_limit'
]

def clean_currency(val):
    if pd.isna(val) or str(val).strip() == '':
        return np.nan

    val = str(val).strip().lower()
    val = val.replace('rp', '')
    val = val.replace('.', '')
    val = val.replace(',', '')
    val = val.replace(' ', '')

    try:
        return float(val)
    except:
        return np.nan

for col in numeric_cols:
    df_clean[col] = df_clean[col].apply(clean_currency)

# cek hasil cleaning numerik
print(df_clean[numeric_cols].head())

# cek missing values setelah cleaning
print('\nMissing values setelah cleaning:')
print(df_clean[numeric_cols].isnull().sum())

   needs_amount  wants_amount  investment_amount  income_amount  budget_limit
0     3400000.0     2250000.0          1600000.0      7250000.0     5650000.0
1      650000.0      400000.0           300000.0      1400000.0     1050000.0
2     2150000.0     1150000.0           550000.0      3800000.0     3300000.0
3      600000.0      350000.0           250000.0      1200000.0      950000.0
4     2550000.0     1450000.0           950000.0      4950000.0     4000000.0

Missing values setelah cleaning:
needs_amount         346
wants_amount         356
investment_amount    495
income_amount        423
budget_limit           0
dtype: int64


In [15]:
# Standarisasi kolom income_date menjadi datetime

def parse_date(val):
    if pd.isna(val) or str(val).strip() == '':
        return pd.NaT

    for fmt in (
        '%Y-%m-%d %H:%M',
        '%Y-%m-%d %H:%M:%S',
        '%Y/%m/%d %H:%M:%S',
        '%d/%m/%Y %H:%M'
    ):
        try:
            return pd.to_datetime(val, format=fmt)
        except:
            continue

    return pd.NaT

df_clean['income_date'] = df_clean['income_date'].apply(parse_date)

before = len(df_clean)

df_clean.dropna(subset=['income_date'], inplace=True)

print(f'Baris tanggal invalid dihapus : {before - len(df_clean)}')
print(f'Tipe kolom income_date : {df_clean["income_date"].dtype}')

Baris tanggal invalid dihapus : 439
Tipe kolom income_date : datetime64[ns]


In [16]:
# Standarisasi kolom source
df_clean['source'] = (
    df_clean['source']
    .astype(str)
    .str.strip()
    .str.lower()
)

# ubah string 'nan' menjadi NaN asli
df_clean['source'] = df_clean['source'].replace('nan', np.nan)

# cari modus source
modus_source = df_clean['source'].mode()[0]

# isi missing value dengan modus
df_clean['source'] = df_clean['source'].fillna(modus_source)

print(df_clean['source'].unique())

['salary' 'scholarship' 'freelance' 'bonus' 'business' 'allowance'
 'part_time' 'internship' 'investment_return']


In [17]:
# Hapus data duplikat
before = len(df_clean)

df_clean.drop_duplicates(inplace=True)

print(f'Duplikat dihapus : {before - len(df_clean)}')
print(f'Sisa data : {len(df_clean)}')

Duplikat dihapus : 37
Sisa data : 9564


In [18]:
# Validasi budget tidak melebihi income
df_clean = df_clean[
    (
        df_clean['needs_amount'] +
        df_clean['wants_amount'] +
        df_clean['investment_amount']
    ) <= df_clean['income_amount']
].copy()

print("Dataset setelah validasi budget:", df_clean.shape)

Dataset setelah validasi budget: (7073, 10)


In [19]:
# hitung ulang budget_limit
df_clean['budget_limit'] = (
    df_clean['needs_amount'] +
    df_clean['wants_amount']
)

In [20]:
print('RINGKASAN SETELAH CLEANING')
print(f'Shape              : {df_clean.shape}')
print(f'Missing values     : {df_clean.isnull().sum().sum()}')
print(f'Duplikat           : {df_clean.duplicated().sum()}')
print(f'Tipe income_date   : {df_clean["income_date"].dtype}')

RINGKASAN SETELAH CLEANING
Shape              : (7073, 10)
Missing values     : 0
Duplikat           : 0
Tipe income_date   : datetime64[ns]


In [21]:
# preview dataset bersih

df_clean.head(10)

,budget_id,user_id,income_id,needs_amount,wants_amount,investment_amount,income_amount,budget_limit,source,income_date
0,BGT01352,USR053,INC06728,3400000.0,2250000.0,1600000.0,7250000.0,5650000.0,salary,2023-01-10 00:38:00
1,BGT06784,USR016,INC00797,650000.0,400000.0,300000.0,1400000.0,1050000.0,scholarship,2024-05-25 12:49:00
3,BGT02948,USR013,INC08009,600000.0,350000.0,250000.0,1200000.0,950000.0,scholarship,2023-07-21 06:16:00
4,BGT03140,USR038,INC06623,2550000.0,1450000.0,950000.0,4950000.0,4000000.0,freelance,2024-04-07 04:12:00
6,BGT01706,USR026,INC01374,1900000.0,1150000.0,650000.0,3700000.0,3050000.0,freelance,2024-11-20 09:55:00
7,BGT01104,USR068,INC06927,4350000.0,2700000.0,1650000.0,8700000.0,7050000.0,freelance,2024-04-30 16:24:00
8,BGT07466,USR040,INC00896,1500000.0,1100000.0,650000.0,3300000.0,2600000.0,freelance,2024-11-29 03:02:00
10,BGT02716,USR064,INC07291,4700000.0,2550000.0,1300000.0,8550000.0,7250000.0,bonus,2023-03-31 09:31:00
11,BGT09094,USR013,INC09837,650000.0,500000.0,300000.0,1450000.0,1150000.0,scholarship,2023-06-23 12:06:00
12,BGT07765,USR059,INC03771,1950000.0,1350000.0,1150000.0,4450000.0,3300000.0,business,2024-09-13 00:58:00


In [22]:
# simpan dataset bersih
df_clean.to_csv("budget_management_clean.csv", index=False)

## Exploratory Data Analysis

### Merged dataset

In [23]:
# Menggabungkan dataset utama transaksi dengan dataset budget

# data budget
budget_id = "19n8KVP-1VJfF3rj6Lzak79U1U7C10LVZ"
budget_url = f"https://drive.google.com/uc?export=download&id={budget_id}"

budget = pd.read_csv(budget_url)

# data transaksi
transactions_id = "1dj122OZA7SmNCpwcSoXaEGE9PY6rlEn5"
transactions_url = f"https://drive.google.com/uc?export=download&id={transactions_id}"

transactions = pd.read_csv(transactions_url)


In [24]:
# Cek isi dataset
budget.head()
budget.info()

transactions.head()
transactions.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7073 entries, 0 to 7072
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   budget_id          7073 non-null   object 
 1   user_id            7073 non-null   object 
 2   income_id          7073 non-null   object 
 3   needs_amount       7073 non-null   float64
 4   wants_amount       7073 non-null   float64
 5   investment_amount  7073 non-null   float64
 6   income_amount      7073 non-null   float64
 7   budget_limit       7073 non-null   float64
 8   source             7073 non-null   object 
 9   income_date        7073 non-null   object 
dtypes: float64(5), object(5)
memory usage: 552.7+ KB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9579 entries, 0 to 9578
Data columns (total 9 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   transaction_id    9579 non-null   object 
 1   user_id    

In [25]:
# Convert dataset

# dataset budget
budget['income_date'] = pd.to_datetime(budget['income_date'])

# dataset transaksi
transactions['date'] = pd.to_datetime(transactions['date'])

In [26]:
# Buat kolom month & year Budget
budget['month'] = budget['income_date'].dt.month
budget['year'] = budget['income_date'].dt.year

In [27]:
# Buat kolom month & year Transactions
transactions['month'] = transactions['date'].dt.month
transactions['year'] = transactions['date'].dt.year

In [28]:
# Merged dataset
merged_df = transactions.merge(
    budget,
    on=['user_id', 'month', 'year'],
    how='left'
)

In [29]:
# Cek hasil merged
merged_df.head()

,transaction_id,user_id,date,merchant,amount,category_detail,category_primary,payment_method,payment_media,month,year,budget_id,income_id,needs_amount,wants_amount,investment_amount,income_amount,budget_limit,source,income_date
0,TXN000001,USR055,2024-10-03 20:33:48,Santika Hotel,685500.0,travel,Wants,Transfer,BSI,10,2024,BGT01276,INC09239,1650000.0,850000.0,600000.0,3100000.0,2500000.0,business,2024-10-16 08:15:00
1,TXN000001,USR055,2024-10-03 20:33:48,Santika Hotel,685500.0,travel,Wants,Transfer,BSI,10,2024,BGT04664,INC09294,2850000.0,1700000.0,1500000.0,6100000.0,4550000.0,salary,2024-10-29 21:39:00
2,TXN000001,USR055,2024-10-03 20:33:48,Santika Hotel,685500.0,travel,Wants,Transfer,BSI,10,2024,BGT05905,INC04571,1900000.0,1050000.0,900000.0,3850000.0,2950000.0,freelance,2024-10-04 11:23:00
3,TXN000001,USR055,2024-10-03 20:33:48,Santika Hotel,685500.0,travel,Wants,Transfer,BSI,10,2024,BGT04855,INC09746,2950000.0,2050000.0,1250000.0,6250000.0,5000000.0,bonus,2024-10-13 12:40:00
4,TXN000001,USR055,2024-10-03 20:33:48,Santika Hotel,685500.0,travel,Wants,Transfer,BSI,10,2024,BGT02705,INC01067,4000000.0,2350000.0,1300000.0,7650000.0,6350000.0,salary,2024-10-09 08:29:00


In [30]:
# Informasi dataset
merged_df.info()
merged_df.isnull().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 37829 entries, 0 to 37828
Data columns (total 20 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   transaction_id     37829 non-null  object        
 1   user_id            37829 non-null  object        
 2   date               37829 non-null  datetime64[ns]
 3   merchant           37829 non-null  object        
 4   amount             37829 non-null  float64       
 5   category_detail    37829 non-null  object        
 6   category_primary   37829 non-null  object        
 7   payment_method     37829 non-null  object        
 8   payment_media      37829 non-null  object        
 9   month              37829 non-null  int32         
 10  year               37829 non-null  int32         
 11  budget_id          37526 non-null  object        
 12  income_id          37526 non-null  object        
 13  needs_amount       37526 non-null  float64       
 14  wants_

,0
transaction_id,0
user_id,0
date,0
merchant,0
amount,0
category_detail,0
category_primary,0
payment_method,0
payment_media,0
month,0


In [31]:
# Cek duplikasi
merged_df.duplicated().sum()

np.int64(0)

In [32]:
# Menghapus row yang ada nilai null
merged_df = merged_df.dropna()

In [33]:
# Cek kembali
merged_df.isnull().sum()

,0
transaction_id,0
user_id,0
date,0
merchant,0
amount,0
category_detail,0
category_primary,0
payment_method,0
payment_media,0
month,0


## Feature Engineering

### Feature Engineering — Cumulative Features

Pada tahap ini dilakukan pembuatan fitur kumulatif (cumulative features) untuk menganalisis pola pengeluaran pengguna secara historis berdasarkan urutan waktu transaksi.

Fitur ini digunakan untuk:
- Melacak total pengeluaran pengguna dari waktu ke waktu
- Mengidentifikasi pola perilaku finansial
- Menjadi dasar perhitungan budget utilization dan overspending detection
- Mendukung behavior pattern analysis dan financial scoring

Feature yang dibuat:
1. `transaction_count_to_date`
   - Jumlah transaksi pengguna hingga transaksi saat ini.

2. `total_expense_to_date`
   - Total akumulasi pengeluaran pengguna hingga transaksi saat ini.

3. `needs_expense_to_date`
   - Total pengeluaran kategori Needs hingga transaksi saat ini.

4. `wants_expense_to_date`
   - Total pengeluaran kategori Wants hingga transaksi saat ini.

5. `investment_expense_to_date`
   - Total pengeluaran kategori Investment hingga transaksi saat ini.

Teknik yang digunakan:
- `groupby()`
- `cumsum()`
- `cumcount()`

Fitur-fitur ini menjadi fondasi utama untuk analisis perilaku finansial dan pengembangan model prediksi overspending.

In [34]:
# Transaction Count To Date (Jumlah transaksi user sampai transaksi saat ini)

merged_df['transaction_count_to_date'] = (
    merged_df.groupby(['user_id', 'year', 'month'])
    .cumcount() + 1
)

In [35]:
# Total Expense To Date (Total pengeluaran user sampai transaksi saat ini)

merged_df['total_expense_to_date'] = (
    merged_df.groupby(['user_id', 'year', 'month'])['amount']
    .cumsum()
)

In [36]:
# Needs Expense To Date (Total pengeluaran kategori Needs sampai transaksi saat ini)

merged_df['needs_expense'] = merged_df['amount'].where(
    merged_df['category_primary'] == 'Needs',
    0
)

merged_df['needs_expense_to_date'] = (
    merged_df.groupby(['user_id', 'year', 'month'])['needs_expense']
    .cumsum()
)

In [37]:
# Wants Expense To Date
merged_df['wants_expense'] = merged_df['amount'].where(
    merged_df['category_primary'] == 'Wants',
    0
)

merged_df['wants_expense_to_date'] = (
    merged_df.groupby(['user_id', 'year', 'month'])['wants_expense']
    .cumsum()
)

In [38]:
# Investment Expense To Date
merged_df['investment_expense'] = merged_df['amount'].where(
    merged_df['category_primary'] == 'Investment',
    0
)

merged_df['investment_expense_to_date'] = (
    merged_df.groupby(['user_id', 'year', 'month'])['investment_expense']
    .cumsum()
)

In [39]:
# Cek hasil 5 feature engineering yang telah dibuat di atas
merged_df[[
    'user_id',
    'date',
    'amount',
    'category_primary',
    'transaction_count_to_date',
    'total_expense_to_date',
    'needs_expense_to_date',
    'wants_expense_to_date',
    'investment_expense_to_date'
]].head(10)

,user_id,date,amount,category_primary,transaction_count_to_date,total_expense_to_date,needs_expense_to_date,wants_expense_to_date,investment_expense_to_date
0,USR055,2024-10-03 20:33:48,685500.0,Wants,1,685500.0,0.0,685500.0,0.0
1,USR055,2024-10-03 20:33:48,685500.0,Wants,2,1371000.0,0.0,1371000.0,0.0
2,USR055,2024-10-03 20:33:48,685500.0,Wants,3,2056500.0,0.0,2056500.0,0.0
3,USR055,2024-10-03 20:33:48,685500.0,Wants,4,2742000.0,0.0,2742000.0,0.0
4,USR055,2024-10-03 20:33:48,685500.0,Wants,5,3427500.0,0.0,3427500.0,0.0
5,USR055,2024-10-03 20:33:48,685500.0,Wants,6,4113000.0,0.0,4113000.0,0.0
6,USR055,2024-10-03 20:33:48,685500.0,Wants,7,4798500.0,0.0,4798500.0,0.0
7,USR055,2024-10-03 20:33:48,685500.0,Wants,8,5484000.0,0.0,5484000.0,0.0
8,USR064,2024-12-03 05:09:38,166000.0,Wants,1,166000.0,0.0,166000.0,0.0
9,USR064,2024-12-03 05:09:38,166000.0,Wants,2,332000.0,0.0,332000.0,0.0


### Feature Engineering — Budget Usage Ratio Features

Pada tahap ini dilakukan pembuatan fitur rasio penggunaan budget (budget usage ratio features) untuk mengukur seberapa besar pengeluaran pengguna dibandingkan dengan budget yang telah ditentukan.

Fitur ini penting untuk:
- Mengidentifikasi potensi overspending
- Mengukur tingkat penggunaan budget pengguna
- Mendukung financial health scoring
- Menjadi dasar sistem alert dan recommendation

Feature yang dibuat:
1. `needs_usage_ratio`
   - Rasio penggunaan budget kategori Needs.

2. `wants_usage_ratio`
   - Rasio penggunaan budget kategori Wants.

3. `investment_usage_ratio`
   - Rasio penggunaan budget kategori Investment.

4. `total_budget_usage_ratio`
   - Rasio total pengeluaran terhadap total budget pengguna.

Interpretasi ratio:
- `< 1`  → Pengeluaran masih dalam batas budget
- `= 1`  → Budget telah digunakan sepenuhnya
- `> 1`  → Terjadi overspending

Teknik yang digunakan:
- Perhitungan rasio berbasis cumulative spending
- Feature scaling berbasis budget allocation

Fitur ini digunakan sebagai komponen utama dalam analisis perilaku finansial, predictive spending alert, dan financial health evaluation.

In [40]:
# Needs Usage Ratio (Seberapa besar budget needs yang udah kepake)

merged_df['needs_usage_ratio'] = (
    merged_df['needs_expense_to_date'] /
    merged_df['needs_amount']
)

In [41]:
# Wants Usage Ratio
merged_df['wants_usage_ratio'] = (
    merged_df['wants_expense_to_date'] /
    merged_df['wants_amount']
)

In [42]:
# Investment Usage Ratio
merged_df['investment_usage_ratio'] = (
    merged_df['investment_expense_to_date'] /
    merged_df['investment_amount']
)

In [43]:
# Total Budget Usage Ratio (Total spending dibanding total budget)

merged_df['total_budget_usage_ratio'] = (
    merged_df['total_expense_to_date'] /
    merged_df['budget_limit']
)

In [44]:
# Cek hasil
merged_df[[
    'needs_usage_ratio',
    'wants_usage_ratio',
    'investment_usage_ratio',
    'total_budget_usage_ratio'
]].describe()

,needs_usage_ratio,wants_usage_ratio,investment_usage_ratio,total_budget_usage_ratio
count,37526.000000,37526.000000,37526.000000,37526.000000
mean,0.742777,4.152449,2.708200,2.627298
std,1.266726,8.137568,7.501861,4.524513
min,-6.490000,-29.211429,0.000000,-91.192857
25%,0.031351,0.110369,0.000000,0.401648
50%,0.305455,1.317037,0.000000,1.271328
75%,0.909758,4.549727,1.338142,3.221570
max,18.216000,126.880000,141.184000,55.088421


### Feature Engineering — Temporal Features

Pada tahap ini dilakukan pembuatan fitur berbasis waktu (temporal features) untuk menganalisis pola transaksi pengguna berdasarkan aspek temporal seperti hari, minggu, dan jarak waktu terhadap pemasukan.

Fitur temporal sangat penting untuk:
- Mengidentifikasi kebiasaan pengeluaran pengguna
- Mendeteksi pola konsumsi berdasarkan waktu
- Mendukung behavior pattern detection
- Membantu model predictive spending alert

Feature yang dibuat:
1. `day_of_week`
   - Hari transaksi dilakukan.

2. `is_weekend`
   - Menandai apakah transaksi terjadi di akhir pekan.

3. `week_of_month`
   - Minggu keberapa transaksi terjadi dalam satu bulan.

4. `days_since_income`
   - Jumlah hari sejak pengguna menerima pemasukan terakhir.

5. `daily_spending_velocity`
   - Kecepatan rata-rata pengeluaran pengguna per hari.

Teknik yang digunakan:
- Ekstraksi datetime
- Perhitungan selisih waktu
- Feature derivation berbasis transaksi historis

Fitur temporal membantu sistem memahami pola pengeluaran pengguna secara lebih kontekstual dan dinamis.

In [45]:
# days_elapsed  (Hari keberapa dalam bulan saat transaksi terjadi)

merged_df['days_elapsed'] = (
    merged_df['date'].dt.day
)

In [46]:
# avg_daily_spending (Rata-rata pengeluaran harian sampai hari transaksi)

merged_df['days_elapsed'] = merged_df['date'].dt.day

merged_df['avg_daily_spending'] = (
    merged_df['total_expense_to_date'] /
    merged_df['days_elapsed']
)

In [47]:
# remaining_days (Sisa hari dalam bulan transaksi)

import calendar

merged_df['days_in_month'] = merged_df.apply(
    lambda x: calendar.monthrange(
        x['year'],
        x['month']
    )[1],
    axis=1
)

merged_df['remaining_days'] = (
    merged_df['days_in_month'] -
    merged_df['days_elapsed']
)

In [48]:
# rolling_7d_spending (Total spending 7 hari terakhir)

merged_df = merged_df.sort_values(
    by=['user_id', 'date']
)

merged_df['rolling_7d_spending'] = (
    merged_df.groupby('user_id')['amount']
    .rolling(window=7, min_periods=1)
    .sum()
    .reset_index(level=0, drop=True)
)

In [49]:
# rolling_30d_spending (Total spending 30 hari terakhir)

merged_df['rolling_30d_spending'] = (
    merged_df.groupby('user_id')['amount']
    .rolling(window=30, min_periods=1)
    .sum()
    .reset_index(level=0, drop=True)
)

In [50]:
# last_month_expense (Total pengeluaran bulan sebelumnya)

monthly_expense = (
    merged_df.groupby(
        ['user_id', 'year', 'month']
    )['amount']
    .sum()
    .reset_index()
)

monthly_expense['last_month_expense'] = (
    monthly_expense.groupby('user_id')['amount']
    .shift(1)
)

merged_df = merged_df.merge(
    monthly_expense[
        ['user_id', 'year', 'month', 'last_month_expense']
    ],
    on=['user_id', 'year', 'month'],
    how='left'
)

In [51]:
# is_overspending (TARGET - Boolean target)

merged_df['is_overspending'] = (
    merged_df['total_budget_usage_ratio'] > 1
)

In [52]:
# overspending_status (Readable status)

def overspending_status(row):

    if row['total_budget_usage_ratio'] > 1:
        return 'overspending'

    elif row['total_budget_usage_ratio'] >= 0.8:
        return 'near_total_limit'

    elif row['wants_usage_ratio'] >= 0.8:
        return 'near_wants_limit'

    else:
        return 'safe'


merged_df['overspending_status'] = (
    merged_df.apply(
        overspending_status,
        axis=1
    )
)

## Final Result

In [53]:
# Cek kolom
merged_df.columns

Index(['transaction_id', 'user_id', 'date', 'merchant', 'amount',
       'category_detail', 'category_primary', 'payment_method',
       'payment_media', 'month', 'year', 'budget_id', 'income_id',
       'needs_amount', 'wants_amount', 'investment_amount', 'income_amount',
       'budget_limit', 'source', 'income_date', 'transaction_count_to_date',
       'total_expense_to_date', 'needs_expense', 'needs_expense_to_date',
       'wants_expense', 'wants_expense_to_date', 'investment_expense',
       'investment_expense_to_date', 'needs_usage_ratio', 'wants_usage_ratio',
       'investment_usage_ratio', 'total_budget_usage_ratio', 'days_elapsed',
       'avg_daily_spending', 'days_in_month', 'remaining_days',
       'rolling_7d_spending', 'rolling_30d_spending', 'last_month_expense',
       'is_overspending', 'overspending_status'],
      dtype='object')

In [54]:
# Preview feature engineering
merged_df[[
    'user_id',
    'date',
    'amount',
    'category_primary',
    'transaction_count_to_date',
    'total_expense_to_date',
    'needs_usage_ratio',
    'wants_usage_ratio',
    'total_budget_usage_ratio',
    'avg_daily_spending',
    'rolling_7d_spending',
    'rolling_30d_spending',
    'last_month_expense',
    'days_elapsed',
    'remaining_days',
    'is_overspending',
    'overspending_status'
]].head(20)

,user_id,date,amount,category_primary,transaction_count_to_date,total_expense_to_date,needs_usage_ratio,wants_usage_ratio,total_budget_usage_ratio,avg_daily_spending,rolling_7d_spending,rolling_30d_spending,last_month_expense,days_elapsed,remaining_days,is_overspending,overspending_status
0,USR001,2023-01-04 07:28:49,1835000.0,Investment,17,12361000.0,1.896000,4.269091,9.508462,3.090250e+06,1835000.0,1835000.0,NaN,4,27,True,overspending
1,USR001,2023-01-04 07:28:49,1835000.0,Investment,18,14196000.0,2.844000,7.826667,17.745000,3.549000e+06,3670000.0,3670000.0,NaN,4,27,True,overspending
2,USR001,2023-01-04 07:28:49,1835000.0,Investment,19,16031000.0,1.015714,2.348000,6.679583,4.007750e+06,5505000.0,5505000.0,NaN,4,27,True,overspending
3,USR001,2023-01-04 07:28:49,1835000.0,Investment,20,17866000.0,2.585455,7.826667,21.018824,4.466500e+06,7340000.0,7340000.0,NaN,4,27,True,overspending
4,USR001,2023-01-11 02:49:16,18000.0,Needs,13,10472000.0,1.824000,4.269091,8.055385,9.520000e+05,7358000.0,7358000.0,NaN,11,20,True,overspending
5,USR001,2023-01-11 02:49:16,18000.0,Needs,14,10490000.0,2.772000,7.826667,13.112500,9.536364e+05,7376000.0,7376000.0,NaN,11,20,True,overspending
6,USR001,2023-01-11 02:49:16,18000.0,Needs,15,10508000.0,1.002857,2.348000,4.378333,9.552727e+05,7394000.0,7394000.0,NaN,11,20,True,overspending
7,USR001,2023-01-11 02:49:16,18000.0,Needs,16,10526000.0,2.585455,7.826667,12.383529,9.569091e+05,5577000.0,7412000.0,NaN,11,20,True,overspending
8,USR001,2023-01-21 08:23:27,587000.0,Wants,5,7343000.0,0.000000,1.067273,5.648462,3.496667e+05,4329000.0,7999000.0,NaN,21,10,True,overspending
9,USR001,2023-01-21 08:23:27,587000.0,Wants,6,7930000.0,0.000000,3.913333,9.912500,3.776190e+05,3081000.0,8586000.0,NaN,21,10,True,overspending


In [55]:
# Preview feature engineering dari akhir
merged_df[[
    'user_id',
    'date',
    'amount',
    'category_primary',
    'transaction_count_to_date',
    'total_expense_to_date',
    'needs_usage_ratio',
    'wants_usage_ratio',
    'total_budget_usage_ratio',
    'avg_daily_spending',
    'rolling_7d_spending',
    'rolling_30d_spending',
    'last_month_expense',
    'days_elapsed',
    'remaining_days',
    'is_overspending',
    'overspending_status'
]].tail(20)

,user_id,date,amount,category_primary,transaction_count_to_date,total_expense_to_date,needs_usage_ratio,wants_usage_ratio,total_budget_usage_ratio,avg_daily_spending,rolling_7d_spending,rolling_30d_spending,last_month_expense,days_elapsed,remaining_days,is_overspending,overspending_status
37506,USR075,2024-12-01 11:53:16,186000.0,Wants,14,7479000.0,0.138571,6.250435,2.301231,7.479000e+06,1542500.0,9951500.0,3738000.0,1,30,True,overspending
37507,USR075,2024-12-01 11:53:16,186000.0,Wants,15,7665000.0,0.098644,5.085517,1.742045,7.665000e+06,1700500.0,9815500.0,3738000.0,1,30,True,overspending
37508,USR075,2024-12-03 16:17:13,110000.0,Needs,19,8292500.0,0.143214,5.091290,1.906322,2.764167e+06,1782500.0,9603500.0,3738000.0,3,28,True,overspending
37509,USR075,2024-12-03 16:17:13,110000.0,Needs,20,8402500.0,0.243333,6.862174,2.585385,2.800833e+06,1521000.0,9391500.0,3738000.0,3,28,True,overspending
37510,USR075,2024-12-03 16:17:13,110000.0,Needs,21,8512500.0,0.210508,5.442414,1.934659,2.837500e+06,1259500.0,8805000.0,3738000.0,3,28,True,overspending
37511,USR075,2024-12-10 13:43:19,111000.0,Wants,1,111000.0,0.000000,0.071613,0.025517,1.110000e+04,999000.0,8219500.0,3738000.0,10,21,False,safe
37512,USR075,2024-12-10 13:43:19,111000.0,Wants,2,222000.0,0.000000,0.193043,0.068308,2.220000e+04,924000.0,7634000.0,3738000.0,10,21,False,safe
37513,USR075,2024-12-10 13:43:19,111000.0,Wants,3,333000.0,0.000000,0.229655,0.075682,3.330000e+04,849000.0,7048500.0,3738000.0,10,21,False,safe
37514,USR075,2024-12-13 12:06:08,2134000.0,Wants,7,2548000.0,0.000000,1.643871,0.585747,1.960000e+05,2797000.0,8486000.0,3738000.0,13,18,False,near_wants_limit
37515,USR075,2024-12-13 12:06:08,2134000.0,Wants,8,4682000.0,0.000000,4.071304,1.440615,3.601538e+05,4821000.0,9923500.0,3738000.0,13,18,True,overspending


In [56]:
# Cek statistik
merged_df.describe()

,date,amount,month,year,needs_amount,wants_amount,investment_amount,income_amount,budget_limit,income_date,...,wants_usage_ratio,investment_usage_ratio,total_budget_usage_ratio,days_elapsed,avg_daily_spending,days_in_month,remaining_days,rolling_7d_spending,rolling_30d_spending,last_month_expense
count,37526,3.752600e+04,37526.000000,37526.000000,3.752600e+04,3.752600e+04,3.752600e+04,3.752600e+04,3.752600e+04,37526,...,37526.000000,37526.000000,37526.000000,37526.000000,3.752600e+04,37526.000000,37526.000000,3.752600e+04,3.752600e+04,3.593100e+04
mean,2024-01-03 05:31:34.291611136,3.974421e+05,6.608778,2023.497921,2.095255e+06,1.253110e+06,8.502865e+05,1.023026e+07,3.348365e+06,2024-01-03 05:35:39.638117632,...,4.152449,2.708200,2.627298,15.774903,8.602362e+05,30.505916,14.731013,2.767993e+06,1.157419e+07,8.582862e+06
min,2023-01-01 03:54:27,4.000000e+03,1.000000,2023.000000,-4.750000e+06,-3.100000e+06,1.000000e+05,1.000000e+06,-3.200000e+06,2023-01-01 01:41:00,...,-29.211429,0.000000,-91.192857,1.000000,2.045455e+02,28.000000,0.000000,8.000000e+03,8.000000e+03,7.000000e+04
25%,2023-07-08 21:13:40,7.750000e+04,4.000000,2023.000000,1.150000e+06,7.000000e+05,4.500000e+05,2.400000e+06,1.850000e+06,2023-07-07 23:37:00,...,0.110369,0.000000,0.401648,8.000000,9.264733e+04,30.000000,7.000000,1.019500e+06,7.191625e+06,2.803000e+06
50%,2023-12-30 16:09:36,2.110000e+05,7.000000,2023.000000,1.900000e+06,1.150000e+06,7.500000e+05,3.950000e+06,3.050000e+06,2023-12-30 19:01:30,...,1.317037,0.000000,1.271328,16.000000,2.865000e+05,31.000000,15.000000,1.927500e+06,1.071400e+07,6.240500e+06
75%,2024-07-06 01:08:53.500000,4.588750e+05,10.000000,2024.000000,2.950000e+06,1.750000e+06,1.150000e+06,6.050000e+06,4.700000e+06,2024-07-04 22:32:00,...,4.549727,1.338142,3.221570,23.000000,7.577279e+05,31.000000,22.000000,3.652875e+06,1.503238e+07,1.155000e+07
max,2024-12-31 23:28:44,2.498000e+06,12.000000,2024.000000,5.500000e+06,3.400000e+06,2.800000e+06,9.850000e+08,8.750000e+06,2024-12-31 19:14:00,...,126.880000,141.184000,55.088421,31.000000,5.251050e+07,31.000000,30.000000,1.724450e+07,4.455800e+07,7.447500e+07
std,NaN,4.979159e+05,3.436728,0.500002,1.229827e+06,7.462857e+05,4.983291e+05,5.915279e+07,1.880838e+06,NaN,...,8.137568,7.501861,4.524513,8.850780,2.189746e+06,0.729034,8.849490,2.527898e+06,6.008233e+06,8.494381e+06


### Feature Validation & Data Quality Checking

Pada tahap ini dilakukan validasi terhadap hasil feature engineering untuk memastikan:
- Tidak terdapat missing value
- Tidak terdapat nilai negatif yang tidak valid
- Rasio penggunaan budget berada pada skala yang realistis
- Tidak terjadi data leakage pada cumulative features

Beberapa penyesuaian dilakukan pada dataset, seperti:
- Menghapus data dengan budget negatif atau nol
- Memastikan cumulative calculation dilakukan per user dan per periode bulan
- Melakukan pengecekan distribusi feature hasil engineering

Tahap validasi ini penting untuk memastikan dataset siap digunakan pada proses modeling dan analisis lebih lanjut.

In [57]:
# Cek distribusi untuk target
merged_df['is_overspending'].value_counts()

,count
is_overspending,
True,21050
False,16476


In [58]:
merged_df['overspending_status'].value_counts()

,count
overspending_status,
overspending,21050
safe,11290
near_wants_limit,3273
near_total_limit,1913


In [59]:
# Cek ratio overspending

merged_df[
    merged_df['is_overspending'] == True
][[
    'total_budget_usage_ratio',
    'is_overspending'
]].head()

,total_budget_usage_ratio,is_overspending
0,9.508462,True
1,17.745000,True
2,6.679583,True
3,21.018824,True
4,8.055385,True


In [60]:
# Cek ratio logic
merged_df[[
    'needs_usage_ratio',
    'wants_usage_ratio',
    'investment_usage_ratio',
    'total_budget_usage_ratio'
]].describe()

,needs_usage_ratio,wants_usage_ratio,investment_usage_ratio,total_budget_usage_ratio
count,37526.000000,37526.000000,37526.000000,37526.000000
mean,0.742777,4.152449,2.708200,2.627298
std,1.266726,8.137568,7.501861,4.524513
min,-6.490000,-29.211429,0.000000,-91.192857
25%,0.031351,0.110369,0.000000,0.401648
50%,0.305455,1.317037,0.000000,1.271328
75%,0.909758,4.549727,1.338142,3.221570
max,18.216000,126.880000,141.184000,55.088421


In [61]:
merged_df[[
    'amount',
    'budget_limit',
    'total_expense_to_date',
    'total_budget_usage_ratio'
]].head(10)

,amount,budget_limit,total_expense_to_date,total_budget_usage_ratio
0,1835000.0,1300000.0,12361000.0,9.508462
1,1835000.0,800000.0,14196000.0,17.745000
2,1835000.0,2400000.0,16031000.0,6.679583
3,1835000.0,850000.0,17866000.0,21.018824
4,18000.0,1300000.0,10472000.0,8.055385
5,18000.0,800000.0,10490000.0,13.112500
6,18000.0,2400000.0,10508000.0,4.378333
7,18000.0,850000.0,10526000.0,12.383529
8,587000.0,1300000.0,7343000.0,5.648462
9,587000.0,800000.0,7930000.0,9.912500


In [62]:
merged_df[merged_df['amount'] < 0][[
    'amount',
    'category_primary',
    'merchant'
]].head(20)

,amount,category_primary,merchant


In [63]:
merged_df[[
    'needs_amount',
    'wants_amount',
    'investment_amount',
    'budget_limit'
]].describe()

,needs_amount,wants_amount,investment_amount,budget_limit
count,3.752600e+04,3.752600e+04,3.752600e+04,3.752600e+04
mean,2.095255e+06,1.253110e+06,8.502865e+05,3.348365e+06
std,1.229827e+06,7.462857e+05,4.983291e+05,1.880838e+06
min,-4.750000e+06,-3.100000e+06,1.000000e+05,-3.200000e+06
25%,1.150000e+06,7.000000e+05,4.500000e+05,1.850000e+06
50%,1.900000e+06,1.150000e+06,7.500000e+05,3.050000e+06
75%,2.950000e+06,1.750000e+06,1.150000e+06,4.700000e+06
max,5.500000e+06,3.400000e+06,2.800000e+06,8.750000e+06


In [64]:
merged_df[
    (merged_df['budget_limit'] <= 0)
][[
    'budget_limit',
    'needs_amount',
    'wants_amount',
    'investment_amount'
]].head()

,budget_limit,needs_amount,wants_amount,investment_amount
222,-250000.0,-650000.0,400000.0,300000.0
228,-250000.0,-650000.0,400000.0,300000.0
234,-250000.0,-650000.0,400000.0,300000.0
240,-250000.0,-650000.0,400000.0,300000.0
246,-250000.0,-650000.0,400000.0,300000.0


In [65]:
merged_df[
    (merged_df['needs_amount'] <= 0) |
    (merged_df['wants_amount'] <= 0) |
    (merged_df['investment_amount'] <= 0)
][[
    'needs_amount',
    'wants_amount',
    'investment_amount'
]].head()

,needs_amount,wants_amount,investment_amount
222,-650000.0,400000.0,300000.0
228,-650000.0,400000.0,300000.0
234,-650000.0,400000.0,300000.0
240,-650000.0,400000.0,300000.0
246,-650000.0,400000.0,300000.0


In [66]:
merged_df = merged_df[
    (merged_df['budget_limit'] > 0) &
    (merged_df['needs_amount'] > 0) &
    (merged_df['wants_amount'] > 0) &
    (merged_df['investment_amount'] > 0)
]

In [67]:
merged_df[[
    'needs_usage_ratio',
    'wants_usage_ratio',
    'investment_usage_ratio',
    'total_budget_usage_ratio'
]].describe()

,needs_usage_ratio,wants_usage_ratio,investment_usage_ratio,total_budget_usage_ratio
count,36783.000000,36783.000000,36783.000000,36783.000000
mean,0.758876,4.232279,2.717368,2.693528
std,1.265695,8.162021,7.491668,4.108726
min,0.000000,0.000000,0.000000,0.000692
25%,0.036375,0.134868,0.000000,0.415364
50%,0.312500,1.357143,0.000000,1.284706
75%,0.922738,4.601701,1.338137,3.213167
max,18.216000,126.880000,141.184000,55.088421


In [68]:
# simpan dataset untuk modeling
merged_df.to_csv("data_budget_modelling.csv", index=False)